In [1]:
# create the directory structure
!mkdir -p MLChurn/data/raw
!mkdir -p MLChurn/data/procesed
!mkdir -p MLChurn/src
# MLChurn/data/raw - capture the raw data
# MLChurn/data/procesed - store clean data
# MLChurn/src - store code files

# create empty files
!touch MLChurn/src/ingest.py
!touch MLChurn/src/preprocess.py
!touch MLChurn/src/train.py

In [2]:
!touch MLChurn/requirements.txt

In [3]:
%%writefile /content/MLChurn/requirements.txt
mlflow
pyngrok

Overwriting /content/MLChurn/requirements.txt


In [4]:
%%writefile /content/MLChurn/src/ingest.py

# to collect the data and store it in our location for further use
def ingest_data():
  import pandas as pd
  data_src = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
  data_dest = "/content/MLChurn/data/raw/"

  df = pd.read_csv(data_src)
  #print(df.head(2))
  df.to_csv(data_dest+'data.csv',index=False)
  print('Data ingestion completed. check folder /content/MLChurn/data/raw')

if __name__ == "__main__":
  ingest_data()

Overwriting /content/MLChurn/src/ingest.py


In [5]:
!python /content/MLChurn/src/ingest.py

Data ingestion completed. check folder /content/MLChurn/data/raw


In [6]:
%%writefile /content/MLChurn/src/preprocess.py

def preprocess_data():
  # set the source and destination path
  import pandas as pd
  data_src = '/content/MLChurn/data/raw'
  data_dest = '/content/MLChurn/data/procesed'

  # load data from source
  df = pd.read_csv(data_src+'/data.csv')
  #print(df.shape)
  #print(df.info())
  #print(df.head(1))

  # to mark the features to ignore for preprocess
  ign_cols = ['customerID','Churn']
  #print(ign_cols)

  # find the category and numerical columns
  cat_cols = df.drop(ign_cols,axis=1).select_dtypes(include='object').columns
  #print(cat_cols)
  num_cols = df.drop(ign_cols,axis=1).select_dtypes(exclude='object').columns
  #print(num_cols)

  #print(len(cat_cols), len(num_cols))

  # check null value and fix it using mean for numeric value and mode for categorical value
  if df[cat_cols].isnull().sum().any():
    df[cat_cols].fillna(df[cat_cols].mode())

  if df[num_cols].isnull().sum().any():
    df[num_cols].fillna(df[num_cols].mean())

  # check and convert target value to numerical
  #print(df['Churn'].unique())
  df['Churn'] = df['Churn'].map({'No':0,'Yes':1})
  #print(df['Churn'].unique())

  # do category encoding
  df_enc = pd.get_dummies(df[cat_cols],columns=cat_cols,drop_first=True,dtype=int)
  #print(df_enc.shape)
  #print(df_enc.head(1))
  #print(df_enc.info())

  # concat all the non-encoded(num_cols)+tgt_cols
  df_new = pd.concat([df[num_cols],df_enc,df['Churn']],axis=1)

  df_new.to_csv(data_dest+'/preprcessed_data.csv',index=False)
  print('Data preprocessing completed.  check folder /content/MLChurn/data/procesed')

if __name__ == "__main__":
  preprocess_data()

Overwriting /content/MLChurn/src/preprocess.py


In [8]:
!pip install -r /content/MLChurn/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 789.2/789.2 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 13.4 MB/s eta 0:00:00


In [110]:
%%writefile /content/MLChurn/src/train.py

import mlflow
import sqlite3
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.tree import DecisionTreeClassifier

def train_model():

  # store tracking info in a light weight db called sqlite
  MLFLOW_TRACKING_URI = 'sqlite:///myprojflow.db'
  mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
  experiment = mlflow.set_experiment('Customer churn Classification on 01 Feb')
  print(experiment.name)

  #experiment = mlflow.set_experiment('Customer churn Classification')
  #print(experiment.name)
  print(experiment.experiment_id)

  data_src = '/content/MLChurn/data/procesed/'
  df = pd.read_csv(data_src+'preprcessed_data.csv',nrows=100)

  print(df.shape)
  print(df.head(1))
  #print(df.info())

  X = df.drop('Churn',axis=1)
  y = df['Churn']

  X_train,X_val,y_train,y_val=train_test_split(X,y,test_size=0.2,random_state=42)
  print(X_train.shape,X_val.shape,y_train.shape,y_val.shape)

  run_name = 'LR L2'
  lm_name = 'lm '+run_name
  with mlflow.start_run(experiment_id = experiment.experiment_id,run_name=run_name ):
    model = LogisticRegression(penalty='l2', solver='lbfgs',max_iter=500)
    #model = LogisticRegression(penalty='l1',solver='saga',max_iter=50)

    model.fit(X_train,y_train)

    pred = model.predict(X_val)
    acc = accuracy_score(y_val,pred)
    f1 = f1_score(y_val,pred)

    #print(acc)
    #print(f1)
    mlflow.log_param('model',run_name)
    print(model)
    if isinstance(model, DecisionTreeClassifier):
      mlflow.log_param('max_depth',model.max_depth)
    elif isinstance(model, LogisticRegression):
      mlflow.log_param('penalty',model.penalty)
      mlflow.log_param('max_iter',model.max_iter)

    mlflow.log_metric('f1_score',f1)
    mlflow.log_metric('acc_score',acc)
    mlflow.sklearn.log_model(model,"model")
if __name__ == "__main__":
  train_model()

Overwriting /content/MLChurn/src/train.py


In [111]:
import mlflow
# store tracking info in a light weight db called sqlite
MLFLOW_TRACKING_URI = 'sqlite:///myprojflow.db'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.set_experiment('Customer churn Classification on 01 Feb')
print(experiment.name)

Customer churn Classification on 01 Feb


In [112]:
import sys
sys.path.append('/content/MLChurn/src') # /content/MLChurn/src
#from MLChurn.src.ingest import ingest_data
import ingest
import preprocess
import train

# to reload the model into current session memory
import importlib
importlib.reload(ingest)
importlib.reload(preprocess)
importlib.reload(train)

# calling/using the method from the module
ingest.ingest_data()
preprocess.preprocess_data()
train.train_model()

Data ingestion completed. check folder /content/MLChurn/data/raw
Data preprocessing completed.  check folder /content/MLChurn/data/procesed
Customer churn Classification on 01 Feb
1
(100, 6560)
   SeniorCitizen  tenure  MonthlyCharges  gender_Male  Partner_Yes  \
0              0       1           29.85            0            1   

   Dependents_Yes  PhoneService_Yes  MultipleLines_No phone service  \
0               0                 0                               1   

   MultipleLines_Yes  InternetService_Fiber optic  ...  TotalCharges_996.45  \
0                  0                            0  ...                    0   

   TotalCharges_996.85  TotalCharges_996.95  TotalCharges_997.65  \
0                    0                    0                    0   

   TotalCharges_997.75  TotalCharges_998.1  TotalCharges_999.45  \
0                    0                   0                    0   

   TotalCharges_999.8  TotalCharges_999.9  Churn  
0                   0                   

2026/02/01 05:25:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


LogisticRegression(max_iter=500)


/usr/local/lib/python3.12/dist-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


In [113]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('myprojflow.db')

e = pd.read_sql_query('SELECT * FROM experiments',conn)
display(e)

m = pd.read_sql_query('SELECT * FROM metrics',conn)
display(m)

p = pd.read_sql_query('SELECT * FROM params',conn)
display(p)

lm = pd.read_sql_query("SELECT * FROM logged_models;", conn)
display(lm)

summ = pd.read_sql_query('''
  SELECT e.name, m.key, MAX(m.value), p.key, p.value, r.name, r.run_uuid
  FROM experiments e
  JOIN runs r on e.experiment_id = r.experiment_id
  JOIN metrics m on r.run_uuid = m.run_uuid AND m.key='f1_score'
  JOIN params p on r.run_uuid = p.run_uuid
  --WHERE #e.experiment_id = m.experiment_id AND e.experiment_id = p.experiment_id
  ''',conn)
display(summ)

summ = pd.read_sql_query('''
  SELECT e.name, m.key, MAX(m.value), p.key, p.value, r.name, r.run_uuid
  FROM experiments e
  JOIN runs r on e.experiment_id = r.experiment_id
  JOIN metrics m on r.run_uuid = m.run_uuid AND m.key='acc_score'
  JOIN params p on r.run_uuid = p.run_uuid
  --WHERE #e.experiment_id = m.experiment_id AND e.experiment_id = p.experiment_id
  ''',conn)

,experiment_id,name,artifact_location,lifecycle_stage,creation_time,last_update_time
0,0,Default,/content/mlruns/0,active,1769919162762,1769919162762
1,1,Customer churn Classification on 01 Feb,/content/mlruns/1,active,1769919162777,1769919162777
2,2,Customer churn Classification,/content/mlruns/2,active,1769919223210,1769919223210


,key,value,timestamp,run_uuid,step,is_nan
0,f1_score,0.666667,1769919262841,cb315d909c8d49b8917afb6d32523256,0,0
1,acc_score,0.800000,1769919262854,cb315d909c8d49b8917afb6d32523256,0,0
2,f1_score,0.666667,1769923515066,479f42569e3c4bbb8e729f7059b41dd5,0,0
3,acc_score,0.800000,1769923515078,479f42569e3c4bbb8e729f7059b41dd5,0,0


,key,value,run_uuid
0,model,LR L2,cb315d909c8d49b8917afb6d32523256
1,penalty,l2,cb315d909c8d49b8917afb6d32523256
2,max_iter,500,cb315d909c8d49b8917afb6d32523256
3,model,LR L2,479f42569e3c4bbb8e729f7059b41dd5
4,penalty,l2,479f42569e3c4bbb8e729f7059b41dd5
5,max_iter,500,479f42569e3c4bbb8e729f7059b41dd5


,model_id,experiment_id,name,artifact_location,creation_timestamp_ms,last_updated_timestamp_ms,status,lifecycle_stage,model_type,source_run_id,status_message
0,m-d6016fd4307c47bca6e4d4b304c8909a,2,model,/content/mlruns/2/models/m-d6016fd4307c47bca6e...,1769919262888,1769919269519,2,active,None,cb315d909c8d49b8917afb6d32523256,None
1,m-5cf445df0f3c4130a822344284cd52a0,1,model,/content/mlruns/1/models/m-5cf445df0f3c4130a82...,1769923515108,1769923519230,2,active,None,479f42569e3c4bbb8e729f7059b41dd5,None


,name,key,MAX(m.value),key,value,name,run_uuid
0,Customer churn Classification,f1_score,0.666667,model,LR L2,LR L2,cb315d909c8d49b8917afb6d32523256


In [91]:
# to pass authtoken access  and start ngrok tunnelling

import getpass

ngrok_token = getpass.getpass("Please enter your ngrok token: ")
get_ipython().system_raw(f'ngrok token {ngrok_token}')
print('Ngrok has been setup')

Please enter your ngrok token: ··········
Ngrok has been setup


In [70]:
get_ipython().system_raw('mlflow ui --backend-store-uri "{MLFLOW_TRACKING_URI}" --host 0.0.0.0 --port 5000 --allowed-hosts "*" &')
get_ipython().system_raw('ngrok http 5000 --log stdout &')

In [164]:
!pkill -f mlflow
!pkill -f ngrok

In [163]:
!lsof -i :5000

COMMAND   PID USER   FD   TYPE  DEVICE SIZE/OFF NODE NAME
python3 38791 root    3u  IPv4 1050125      0t0  TCP *:5000 (LISTEN)
python3 38794 root    3u  IPv4 1050125      0t0  TCP *:5000 (LISTEN)
python3 38795 root    3u  IPv4 1050125      0t0  TCP *:5000 (LISTEN)
python3 38796 root    3u  IPv4 1050125      0t0  TCP *:5000 (LISTEN)
python3 38797 root    3u  IPv4 1050125      0t0  TCP *:5000 (LISTEN)


In [77]:
!curl -s http://localhost:4040/api/tunnels

In [108]:
import subprocess

# Kill any existing ngrok processes
subprocess.run(['killall', 'ngrok'], capture_output=True, text=True)
subprocess.run(['killall', 'mlflow'], capture_output=True, text=True)
print('Attempted to kill ngrok processes.')

Attempted to kill ngrok processes.


In [149]:
# Start ngrok and redirect its output to a log file
!ngrok config add-authtoken 393MNSymAdskCPL5CZIfAnp2Yn9_6vX9U6GMEQqLw2GFE2vVb
with open('ngrok.log', 'w') as log_file:
    get_ipython().system_raw('mlflow ui --backend-store-uri "{MLFLOW_TRACKING_URI}" --host 0.0.0.0 --port 5000 --allowed-hosts "*" &')
    get_ipython().system_raw('ngrok http 5000 --log "ngrok.log" &')
    #get_ipython().system_raw('ngrok http 5000 --log stdout &')
print('Ngrok started and logging to ngrok.log')

# Give ngrok a moment to start and establish the tunnel
import time
time.sleep(5)

# Read the ngrok log file
with open('ngrok.log', 'r') as log_file:
    ngrok_output = log_file.read()

print('\n--- Ngrok Log Output ---')
print(ngrok_output)
print('------------------------')

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Ngrok started and logging to ngrok.log

--- Ngrok Log Output ---
t=2026-02-01T06:00:50+0000 lvl=info msg="no configuration paths supplied"
t=2026-02-01T06:00:50+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
t=2026-02-01T06:00:50+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=nil
t=2026-02-01T06:00:50+0000 lvl=info msg="FIPS 140 mode" enabled=false
t=2026-02-01T06:00:50+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
t=2026-02-01T06:00:50+0000 lvl=info msg="client session established" obj=tunnels.session
t=2026-02-01T06:00:50+0000 lvl=info msg="tunnel session started" obj=tunnels.session
t=2026-02-01T06:00:50+0000 lvl=info msg="started tunnel" obj=tunnels name=command_line addr=http://localhost:5000 url=https://viably-subchorioid-amir.ngrok-free.dev

------------------------


In [150]:
# Try to get the ngrok tunnel URL again
import json
import requests

# Assuming ngrok is running and its API is accessible at 4040
try:
    response = requests.get('http://localhost:4040/api/tunnels')
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    tunnels_data = response.json()

    # Extract and print the public URL if available
    for tunnel in tunnels_data.get('tunnels', []):
        if tunnel.get('proto') == 'https':
            public_url = tunnel.get('public_url')
            if public_url:
                print(f'Ngrok Tunnel URL: {public_url}')
                break
    else:
        print('No public HTTPS tunnel found.')

except requests.exceptions.RequestException as e:
    print(f"Error connecting to ngrok API: {e}")
    print("Please check if ngrok is running and the port 4040 is open.")


Ngrok Tunnel URL: https://viably-subchorioid-amir.ngrok-free.dev


In [117]:
from mlflow import search_runs

runs = mlflow.search_runs(order_by=["metrics.acc_score DESC"])
best_run = runs.iloc[0]
print('best acc-scor:',best_run['metrics.acc_score'])
print('info abt best model')
print('-'*15)
print(best_run)

best acc-scor: 0.8
info abt best model
---------------
run_id                                      479f42569e3c4bbb8e729f7059b41dd5
experiment_id                                                              1
status                                                              FINISHED
artifact_uri               /content/mlruns/1/479f42569e3c4bbb8e729f7059b4...
start_time                                  2026-02-01 05:25:13.886000+00:00
end_time                                    2026-02-01 05:25:19.243000+00:00
metrics.acc_score                                                        0.8
metrics.f1_score                                                    0.666667
params.model                                                           LR L2
params.penalty                                                            l2
params.max_iter                                                          500
tags.mlflow.source.type                                             NOTEBOOK
tags.mlflow.user     

In [118]:
mlflow.register_model(f"runs:/{best_run.run_id}/model", "BestAccModel")

Successfully registered model 'BestAccModel'.
2026/02/01 05:52:29 WARNING mlflow.tracking._model_registry.fluent: Run with id 479f42569e3c4bbb8e729f7059b41dd5 has no artifacts at artifact path 'model', registering model based on models:/m-5cf445df0f3c4130a822344284cd52a0 instead
Created version '1' of model 'BestAccModel'.


<ModelVersion: aliases=[], creation_timestamp=1769925149847, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1769925149847, metrics=None, model_id=None, name='BestAccModel', params=None, run_id='479f42569e3c4bbb8e729f7059b41dd5', run_link=None, source='models:/m-5cf445df0f3c4130a822344284cd52a0', status='READY', status_message=None, tags={}, user_id=None, version=1>

In [119]:
# do predictions from best model


import pandas as pd
import mlflow.pyfunc

model_name  = "BestAccModel"
model_version =1

model = mlflow.pyfunc.load_model(model_uri=f"models:/{model_name}/{model_version}")

# read test data
df_test = pd.read_csv("/content/MLChurn/data/procesed/preprcessed_data.csv",header=0)
print(df_test.shape)
display(df_test.tail(1))
df_test = df_test.tail(1)

testX = df_test.drop('Churn',axis=1)
testY = df_test['Churn']

predictions = model.predict(testX)
print(predictions, testY.values)


(7043, 6560)


,SeniorCitizen,tenure,MonthlyCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,...,TotalCharges_996.45,TotalCharges_996.85,TotalCharges_996.95,TotalCharges_997.65,TotalCharges_997.75,TotalCharges_998.1,TotalCharges_999.45,TotalCharges_999.8,TotalCharges_999.9,Churn
7042,0,66,105.65,1,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0


[0] [0]


In [120]:
#move the selected model to productionize stage
from mlflow.tracking import MlflowClient

client = MlflowClient()

model_name  = "BestAccModel"
model_version =1

client.transition_model_version_stage(name = model_name,  version = model_version, stage = "Production")

/tmp/ipython-input-1417549836.py:9: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(name = model_name,  version = model_version, stage = "Production")


<ModelVersion: aliases=[], creation_timestamp=1769925149847, current_stage='Production', deployment_job_state=None, description=None, last_updated_timestamp=1769925270446, metrics=None, model_id=None, name='BestAccModel', params=None, run_id='479f42569e3c4bbb8e729f7059b41dd5', run_link=None, source='models:/m-5cf445df0f3c4130a822344284cd52a0', status='READY', status_message=None, tags={}, user_id=None, version=1>

In [169]:
get_ipython().system_raw('mlflow models serve -m "models:/BestAccModel/Production" -h 0.0.0.0 -p 5001 --no-conda &')
import time
time.sleep(10) # Give the server time to start

In [167]:
!pkill -f uvicorn

In [171]:
!lsof -i :5001

COMMAND   PID USER   FD   TYPE  DEVICE SIZE/OFF NODE NAME
uvicorn 40010 root   15u  IPv4 1081216      0t0  TCP *:5001 (LISTEN)


In [123]:
!lsof -i :5000

COMMAND   PID USER   FD   TYPE DEVICE SIZE/OFF NODE NAME
python3 30120 root    3u  IPv4 784005      0t0  TCP *:5000 (LISTEN)
python3 30123 root    3u  IPv4 784005      0t0  TCP *:5000 (LISTEN)
python3 30124 root    3u  IPv4 784005      0t0  TCP *:5000 (LISTEN)
python3 30125 root    3u  IPv4 784005      0t0  TCP *:5000 (LISTEN)
python3 30126 root    3u  IPv4 784005      0t0  TCP *:5000 (LISTEN)


In [172]:
from pyngrok import ngrok
import json
import requests

# Ensure all ngrok tunnels are closed before trying to open a new one
# This is a workaround for the 5-tunnel limit on free ngrok accounts.
ngrok.kill()

server_url = ngrok.connect(5001)
print(server_url)
url = server_url.public_url + '/invocations'
print(url)

#print(testX.shape)
#print(testX.head(1))
mydata = {"instances" : testX.values.tolist()}
print(mydata)

response = requests.post(url, json=mydata)
print(response.json())

NgrokTunnel: "https://viably-subchorioid-amir.ngrok-free.dev" -> "http://localhost:5001"
https://viably-subchorioid-amir.ngrok-free.dev/invocations
{'instances': [[0.0, 66.0, 105.65, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.